# Extra experiment: tool-using intent parser (GPU)

Baseline 3 (prompted Qwen2.5-7B, prompt p1-v3) **plus read-only data tools**: before writing the contract the parser may
call `profile_column` (dtype, missing, top values, min/max/mean/median, optionally on rows matching a condition),
`count_rows` and `inspect_rows` on the full input data. Same base prompt, same 5 dev few-shot examples, greedy decoding,
same extraction and strict validation as Baseline 3; prompt version **t1-v1**. Code: `src/igt/agents/tool_parser.py`,
`eval/parse_contracts_tools.py`.

Question: does letting the parser look at the data reduce invalid and wrong contracts (e.g. "fill with the median" written
as the text "median")? It is compared with Baseline 3 on the same 120 tasks; it doesn't replace any system.

**After the run:** download `parse_tools_7b.zip`, unzip it to `kaggle_outputs/parsing/7b_t1-v1/` and run
`.venv/bin/python eval/score_parsers.py` (no arguments) on the laptop. Results go to `results/stage1/tool_parser/`.

Settings: GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.
**GPU time (estimate, not measured):** 1–7 model calls per task with a ~5k-token prompt, roughly 1–3 h for 120 tasks.
Resumable: rerun the notebook and finished tasks are skipped.

In [ ]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True    # the Baseline 3 model
SYSTEM = "intent_tools_7b_t1v1"                                # name used on the laptop
LIMIT = None                    # e.g. 5 for a quick check first
WORK = "/kaggle/working"
OUT = f"{WORK}/parse_tools_7b"
CACHE = f"{WORK}/llm_cache"

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
NEEDED = ["src/igt/agents/tool_parser.py", "eval/parse_contracts_tools.py", "src/igt/llm/hf.py"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas (the tools read the data with it) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers, bitsandbytes; "
         "print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
# ---------------- Parse every task (output streamed live; resumable) ----------------
cmd = ["python", "eval/parse_contracts_tools.py", "--model", MODEL_ID, "--out", OUT, "--cache-dir", CACHE]
cmd += ["--load-in-4bit"] if LOAD_IN_4BIT else []
cmd += ["--limit", str(LIMIT)] if LIMIT else []
t0 = time.time()
p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    if "Loading weights" not in line:
        print(line, end="", flush=True)
assert p.wait() == 0, "parsing failed: see the output above"
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ---------------- Quick look + zip ----------------
R = [json.loads(l) for l in open(f"{OUT}/parses.jsonl")]
print(f"{len(R)} parses | status {dict(collections.Counter(r['parse_status'] for r in R))}")
print(f"tasks that used a tool: {sum(r['n_tool_calls'] > 0 for r in R)} | tool calls {sum(r['n_tool_calls'] for r in R)} | "
      f"forced to answer {sum(r['forced_answer'] for r in R)}")
print("tools:", dict(collections.Counter(c['tool'] for r in R for c in r['tool_calls'])))
open(f"{OUT}/system.txt", "w").write(SYSTEM)
open(f"{OUT}/git_commit.txt", "w").write(COMMIT)
print("saved:", shutil.make_archive(OUT, "zip", OUT))
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)